# 02 - Cleaning, merging, label generation
Inspect raw data, fix sentinels/units, check ranges, merge on date, build features and the `danger_5to7d` label.

In [1]:
import sys; sys.path.insert(0, '..')   # so `src` is importable from notebooks/
import pandas as pd
from src.config import RAW_DIR
om = pd.read_csv(RAW_DIR/'open_meteo.csv'); gs = pd.read_csv(RAW_DIR/'noaa_gsod.csv', low_memory=False)
for d in (om, gs):
    print(d.shape, d.duplicated().sum(), 'duplicates'); display(d.isnull().sum()[lambda s: s>0]); display(d.describe().T.head(12))

(16802, 15) 0 duplicates


Series([], dtype: int64)

,count,mean,std,min,25%,50%,75%,max
temperature_2m_max,16802.0,31.467432,3.860667,16.700000,29.200000,32.00000,34.000000,46.400000
temperature_2m_min,16802.0,22.385776,5.404372,6.300000,18.000000,24.00000,27.000000,33.100000
temperature_2m_mean,16802.0,26.529389,4.205530,11.900000,23.300000,28.00000,29.700000,37.900000
apparent_temperature_max,16802.0,32.852113,5.777040,10.000000,28.700000,34.20000,37.100000,49.600000
apparent_temperature_mean,16802.0,28.253821,6.624227,6.400000,23.000000,30.50000,33.500000,42.500000
precipitation_sum,16802.0,0.402321,3.363800,0.000000,0.000000,0.00000,0.000000,129.800000
wind_speed_10m_max,16802.0,22.832240,6.625323,8.000000,18.000000,22.40000,27.200000,50.500000
wind_gusts_10m_max,16802.0,38.825664,10.099311,13.300000,31.300000,38.50000,45.700000,78.100000
shortwave_radiation_sum,16802.0,20.555742,4.210199,1.580000,17.320000,20.25000,23.880000,30.280000
et0_fao_evapotranspiration,16802.0,5.272848,1.294556,1.090000,4.350000,5.17000,6.150000,13.760000


(15618, 28) 0 duplicates


Series([], dtype: int64)

,count,mean,std,min,25%,50%,75%,max
STATION,15618.0,4.178010e+10,0.000000e+00,4.178010e+10,4.178010e+10,4.178010e+10,4.178010e+10,4.178010e+10
LATITUDE,15618.0,2.490655e+01,7.105655e-15,2.490655e+01,2.490655e+01,2.490655e+01,2.490655e+01,2.490655e+01
LONGITUDE,15618.0,6.716080e+01,0.000000e+00,6.716080e+01,6.716080e+01,6.716080e+01,6.716080e+01,6.716080e+01
ELEVATION,15618.0,3.048000e+01,7.105655e-15,3.048000e+01,3.048000e+01,3.048000e+01,3.048000e+01,3.048000e+01
TEMP,15618.0,8.010108e+01,8.142761e+00,5.140000e+01,7.400000e+01,8.290000e+01,8.630000e+01,1.011000e+02
TEMP_ATTRIBUTES,15618.0,1.594494e+01,8.198163e+00,4.000000e+00,7.000000e+00,2.100000e+01,2.400000e+01,2.400000e+01
DEWP,15618.0,1.012589e+02,6.149026e+02,-1.330000e+01,5.260000e+01,6.850000e+01,7.500000e+01,9.999900e+03
DEWP_ATTRIBUTES,15618.0,1.585011e+01,8.243260e+00,0.000000e+00,7.000000e+00,2.100000e+01,2.400000e+01,2.400000e+01
SLP,15618.0,4.334497e+03,4.340868e+03,9.915000e+02,1.006600e+03,1.015200e+03,9.999900e+03,9.999900e+03
SLP_ATTRIBUTES,15618.0,5.151108e+00,5.520819e+00,0.000000e+00,0.000000e+00,5.000000e+00,8.000000e+00,2.400000e+01


In [2]:
from src.data_loader import build_dataset
ds = build_dataset()   # asserts: >=5000 rows, >=15 features, both classes, unique dates
ds.head()

{
  "open_meteo": {
    "rows_raw": 16802,
    "bad_dates": 0,
    "duplicate_dates": 0,
    "out_of_range_set_nan": {
      "relative_humidity_2m_mean": 0,
      "cloud_cover_mean": 0,
      "temperature_2m_max": 0,
      "temperature_2m_min": 0,
      "surface_pressure_mean": 0,
      "precipitation_sum": 0
    },
    "rows_clean": 16802
  },
  "gsod": {
    "rows_raw": 15618,
    "duplicate_dates": 0,
    "rows_clean": 15618,
    "missing_station_heat_index": 61
  },
  "merged_rows": 15618,
  "final_rows": 14772,
  "n_features": 29,
  "date_min": "1980-01-01",
  "date_max": "2025-08-17",
  "missing_cells_in_features": 7,
  "class_counts": {
    "0": 9749,
    "1": 5023
  },
  "positive_rate": 0.3400352017330084
}


,date,temperature_2m_max,temperature_2m_min,temperature_2m_mean,apparent_temperature_max,apparent_temperature_mean,precipitation_sum,wind_speed_10m_max,wind_gusts_10m_max,shortwave_radiation_sum,...,humidity_roll3,dew_point_roll7,pressure_change_3d,month_sin,month_cos,doy_sin,doy_cos,season,danger_5to7d,station_heat_index_c
0,1980-01-01,20.8,13.2,16.9,18.7,14.4,0.0,16.6,29.2,16.73,...,NaN,NaN,NaN,0.5,0.866025,0.017202,0.999852,winter,0,23.303312
1,1980-01-05,21.4,9.8,15.0,18.2,12.0,0.0,21.3,36.0,16.64,...,60.930556,7.219167,5.320833,0.5,0.866025,0.085906,0.996303,winter,0,21.078829
2,1980-01-06,21.9,9.1,15.0,17.3,11.5,0.0,25.1,41.4,16.66,...,58.305556,6.859722,1.225000,0.5,0.866025,0.103031,0.994678,winter,0,22.256000
3,1980-01-07,23.4,11.1,16.4,19.7,12.9,0.0,21.1,35.6,16.86,...,54.166667,6.572024,-0.791667,0.5,0.866025,0.120126,0.992759,winter,0,23.187812
4,1980-01-08,23.5,9.0,16.1,22.0,14.1,0.0,13.7,24.8,16.39,...,54.430556,6.569643,-2.104167,0.5,0.866025,0.137185,0.990545,winter,0,24.430343
